# Customer Segmentation with K-Means Clustering

This notebook performs customer segmentation using K-means clustering on key behavioral and demographic features. We'll identify distinct customer groups, profile them, and provide actionable insights for targeted marketing.

## Table of Contents
1. Data Loading and Preparation
2. Feature Selection and Standardization
3. Determining Optimal Number of Clusters
4. K-Means Clustering
5. Cluster Profiling and Naming
6. Visualizations
7. Key Insights and Recommendations

## 1. Data Loading and Preparation

In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
px.defaults.template = "plotly_white"

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [66]:
# Load the cleaned dataset
df = pd.read_csv("../data/processed/ifood_df_cleaned.csv")

print(f"Dataset shape: {df.shape}")
print(f"Number of customers: {df.shape[0]:,}")
df.head()

,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,...,marital_Together,marital_Widow,education_Basic,education_PhD,MntTotal,MntRegularProds,AcceptedCmpOverall,education_Masters,education_Bachelors,TotalChildren
0,58138.0,0,0,58,635,88,546,172,88,88,...,0,0,0,0,1529,1441,0,0,1,0
1,46344.0,1,1,38,11,1,6,2,1,6,...,0,0,0,0,21,15,0,0,1,2
2,71613.0,0,0,26,426,49,127,111,21,42,...,1,0,0,0,734,692,0,0,1,0
3,26646.0,1,0,26,11,4,20,10,3,5,...,1,0,0,0,48,43,0,0,1,1
4,58293.0,1,0,94,173,43,118,46,27,15,...,0,0,0,1,407,392,0,0,0,1


## 2. Feature Selection and Standardization

We'll focus on key behavioral and demographic features:
- **Income**: Financial capacity
- **Age**: Life stage
- **Recency**: Recent engagement
- **MntTotal**: Overall spending behavior
- **Kidhome & Teenhome**: Household composition

In [67]:
# Select features for clustering
clustering_features = ['Income', 'Age', 'Recency', 'MntTotal', 'Kidhome', 'Teenhome']

# Create feature dataframe
X = df[clustering_features].copy()

print("Features selected for clustering:")
print(X.describe())

# Check for missing values
print(f"\nMissing values: {X.isnull().sum().sum()}")

Features selected for clustering:
              Income          Age      Recency     MntTotal      Kidhome  \
count    2205.000000  2205.000000  2205.000000  2205.000000  2205.000000   
mean    51622.094785    51.095692    49.009070   562.764626     0.442177   
std     20713.063826    11.705801    28.932111   575.936911     0.537132   
min      1730.000000    24.000000     0.000000     4.000000     0.000000   
25%     35196.000000    43.000000    24.000000    56.000000     0.000000   
50%     51287.000000    50.000000    49.000000   343.000000     0.000000   
75%     68281.000000    61.000000    74.000000   964.000000     1.000000   
max    113734.000000    80.000000    99.000000  2491.000000     2.000000   

          Teenhome  
count  2205.000000  
mean      0.506576  
std       0.544380  
min       0.000000  
25%       0.000000  
50%       0.000000  
75%       1.000000  
max       2.000000  

Missing values: 0


In [68]:
# Standardize features (critical for K-means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to dataframe for easier handling
X_scaled_df = pd.DataFrame(X_scaled, columns=clustering_features)

print("Standardized features:")
print(X_scaled_df.describe())

print("\n✓ Features standardized (mean=0, std=1)")

Standardized features:
             Income           Age       Recency      MntTotal       Kidhome  \
count  2.205000e+03  2.205000e+03  2.205000e+03  2.205000e+03  2.205000e+03   
mean   2.255691e-17  1.643432e-16  7.975480e-17 -3.705778e-17  4.833624e-18   
std    1.000227e+00  1.000227e+00  1.000227e+00  1.000227e+00  1.000227e+00   
min   -2.409272e+00 -2.315248e+00 -1.694318e+00 -9.704038e-01 -8.234051e-01   
25%   -7.932106e-01 -6.917534e-01 -8.646014e-01 -8.800957e-01 -8.234051e-01   
50%   -1.618161e-02 -9.362368e-02 -3.135738e-04 -3.816642e-01 -8.234051e-01   
75%    8.044529e-01  8.462945e-01  8.639742e-01  6.968235e-01  1.038757e+00   
max    2.999363e+00  2.469790e+00  1.728262e+00  3.348757e+00  2.900920e+00   

           Teenhome  
count  2.205000e+03  
mean   1.933450e-17  
std    1.000227e+00  
min   -9.307668e-01  
25%   -9.307668e-01  
50%   -9.307668e-01  
75%    9.066018e-01  
max    2.743970e+00  

✓ Features standardized (mean=0, std=1)


## 3. Determining Optimal Number of Clusters

We'll use two methods:
1. **Elbow Method**: Look for the "elbow" in the inertia plot
2. **Silhouette Score**: Measure how well-separated clusters are

In [69]:
# Elbow Method
inertias = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Plot elbow curve
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(K_range), y=inertias, mode='lines+markers',
                         marker=dict(size=10, color='steelblue'),
                         line=dict(color='steelblue', width=2)))
fig.update_layout(title='Elbow Method: Inertia vs Number of Clusters',
                  xaxis_title='Number of Clusters (k)',
                  yaxis_title='Inertia (Within-Cluster Sum of Squares)',
                  height=500,
                  showlegend=False)
fig.show()

In [70]:
print("Inertia values:")
for k, inertia in zip(K_range, inertias):
    print(f"k={k}: {inertia:.2f}")

Inertia values:
k=2: 9152.81
k=3: 7382.53
k=4: 6251.98
k=5: 5691.59
k=6: 5161.40
k=7: 4722.51
k=8: 4413.25
k=9: 4126.38
k=10: 3889.81


In [71]:
# Silhouette Score Method
silhouette_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)
    silhouette_avg = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(silhouette_avg)

# Plot silhouette scores
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(K_range), y=silhouette_scores, mode='lines+markers',
                         marker=dict(size=10, color='seagreen'),
                         line=dict(color='seagreen', width=2)))
fig.update_layout(title='Silhouette Score vs Number of Clusters',
                  xaxis_title='Number of Clusters (k)',
                  yaxis_title='Silhouette Score',
                  height=500,
                  showlegend=False)
fig.show()

In [72]:
print("Silhouette scores:")
for k, score in zip(K_range, silhouette_scores):
    print(f"k={k}: {score:.4f}")

# Find optimal k
optimal_k = list(K_range)[silhouette_scores.index(max(silhouette_scores))]
print(f"\n✓ Optimal k based on silhouette score: {optimal_k}")

Silhouette scores:
k=2: 0.2857
k=3: 0.2673
k=4: 0.2786
k=5: 0.2670
k=6: 0.2723
k=7: 0.2548
k=8: 0.2545
k=9: 0.2622
k=10: 0.2544

✓ Optimal k based on silhouette score: 2


In [73]:
# Combined visualization
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=('Elbow Method', 'Silhouette Score'))

fig.add_trace(go.Scatter(x=list(K_range), y=inertias, mode='lines+markers',
                         marker=dict(size=8, color='steelblue'),
                         line=dict(color='steelblue', width=2),
                         name='Inertia'), row=1, col=1)

fig.add_trace(go.Scatter(x=list(K_range), y=silhouette_scores, mode='lines+markers',
                         marker=dict(size=8, color='seagreen'),
                         line=dict(color='seagreen', width=2),
                         name='Silhouette'), row=1, col=2)

fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=1)
fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=2)
fig.update_yaxes(title_text="Inertia", row=1, col=1)
fig.update_yaxes(title_text="Silhouette Score", row=1, col=2)

fig.update_layout(height=400, showlegend=False, 
                  title_text="Optimal Cluster Number Analysis")
fig.show()

### Decision on Number of Clusters

Based on the elbow method and silhouette scores, we'll choose **k=4** clusters. This provides:
- Clear separation between groups
- Manageable number of segments for marketing
- Good balance between granularity and interpretability

## 4. K-Means Clustering

In [74]:
# Perform K-means clustering with optimal k
optimal_k = 4  # Based on analysis above

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Calculate final silhouette score
final_silhouette = silhouette_score(X_scaled, df['Cluster'])

print(f"✓ K-means clustering completed with k={optimal_k}")
print(f"Final silhouette score: {final_silhouette:.4f}")
print(f"\nCluster sizes:")
print(df['Cluster'].value_counts().sort_index())
print(f"\nCluster distribution:")
for cluster in sorted(df['Cluster'].unique()):
    count = (df['Cluster'] == cluster).sum()
    pct = count / len(df) * 100
    print(f"Cluster {cluster}: {count} customers ({pct:.1f}%)")

✓ K-means clustering completed with k=4
Final silhouette score: 0.2786

Cluster sizes:
Cluster
0    671
1    608
2    519
3    407
Name: count, dtype: int64

Cluster distribution:
Cluster 0: 671 customers (30.4%)
Cluster 1: 608 customers (27.6%)
Cluster 2: 519 customers (23.5%)
Cluster 3: 407 customers (18.5%)


## 5. Cluster Profiling and Naming

Let's analyze each cluster's characteristics to understand who they are and assign meaningful names.

In [75]:
# Calculate cluster profiles
cluster_profile = df.groupby('Cluster')[clustering_features].mean()
cluster_profile['Count'] = df.groupby('Cluster').size()
cluster_profile['Pct'] = (cluster_profile['Count'] / len(df) * 100).round(1)

# Add additional profiling features
profile_features = ['MntWines', 'MntMeatProducts', 'NumWebPurchases', 
                    'NumStorePurchases', 'AcceptedCmpOverall', 'NumWebVisitsMonth']
for feature in profile_features:
    if feature in df.columns:
        cluster_profile[feature] = df.groupby('Cluster')[feature].mean()

print("Cluster Profiles (Mean Values):")
print("="*100)
print(cluster_profile.round(2))

Cluster Profiles (Mean Values):
           Income    Age  Recency  MntTotal  Kidhome  Teenhome  Count   Pct  \
Cluster                                                                       
0        56608.18  56.93    48.37    602.15     0.01      0.99    671  30.4   
1        30485.39  41.96    48.82    116.78     0.83      0.00    608  27.6   
2        76578.08  51.47    49.83   1343.50     0.05      0.05    519  23.5   
3        43153.53  54.66    49.30    168.48     1.07      1.04    407  18.5   

         MntWines  MntMeatProducts  NumWebPurchases  NumStorePurchases  \
Cluster                                                                  
0          387.66           127.67             5.40               7.13   
1           47.97            37.74             2.51               3.42   
2          658.41           461.64             5.15               8.41   
3          108.33            40.07             2.99               3.96   

         AcceptedCmpOverall  NumWebVisitsMonth  

In [76]:
# Detailed breakdown by cluster
for cluster in sorted(df['Cluster'].unique()):
    print(f"\n{'='*80}")
    print(f"CLUSTER {cluster}")
    print(f"{'='*80}")
    cluster_data = df[df['Cluster'] == cluster]
    
    print(f"Size: {len(cluster_data)} customers ({len(cluster_data)/len(df)*100:.1f}%)\n")
    
    print("Demographics:")
    print(f"  Average Income: ${cluster_data['Income'].mean():,.0f}")
    print(f"  Average Age: {cluster_data['Age'].mean():.1f} years")
    print(f"  Avg Kids at Home: {cluster_data['Kidhome'].mean():.2f}")
    print(f"  Avg Teens at Home: {cluster_data['Teenhome'].mean():.2f}")
    print(f"  Total Children: {(cluster_data['Kidhome'] + cluster_data['Teenhome']).mean():.2f}")
    
    print(f"\nSpending Behavior:")
    print(f"  Total Spending: ${cluster_data['MntTotal'].mean():,.0f}")
    print(f"  Wine Spending: ${cluster_data['MntWines'].mean():,.0f}")
    print(f"  Meat Spending: ${cluster_data['MntMeatProducts'].mean():,.0f}")
    print(f"  Recency: {cluster_data['Recency'].mean():.1f} days")
    
    print(f"\nPurchase Channels:")
    print(f"  Store Purchases: {cluster_data['NumStorePurchases'].mean():.1f}")
    print(f"  Web Purchases: {cluster_data['NumWebPurchases'].mean():.1f}")
    print(f"  Web Visits/Month: {cluster_data['NumWebVisitsMonth'].mean():.1f}")
    
    print(f"\nCampaign Response:")
    print(f"  Avg Campaigns Accepted: {cluster_data['AcceptedCmpOverall'].mean():.2f}")
    print(f"  Campaign Response Rate: {(cluster_data['AcceptedCmpOverall'] > 0).sum()/len(cluster_data)*100:.1f}%")

Size: 608 customers (27.6%)

Demographics:
  Average Income: $30,485
  Average Age: 42.0 years
  Avg Kids at Home: 0.83
  Avg Teens at Home: 0.00
  Total Children: 0.83

Spending Behavior:
  Total Spending: $117
  Wine Spending: $48
  Meat Spending: $38
  Recency: 48.8 days

Purchase Channels:
  Store Purchases: 3.4
  Web Purchases: 2.5
  Web Visits/Month: 6.8

Campaign Response:
  Avg Campaigns Accepted: 0.10
  Campaign Response Rate: 9.4%

CLUSTER 2
Size: 519 customers (23.5%)

Demographics:
  Average Income: $76,578
  Average Age: 51.5 years
  Avg Kids at Home: 0.05
  Avg Teens at Home: 0.05
  Total Children: 0.10

Spending Behavior:
  Total Spending: $1,343
  Wine Spending: $658
  Meat Spending: $462
  Recency: 49.8 days

Purchase Channels:
  Store Purchases: 8.4
  Web Purchases: 5.2
  Web Visits/Month: 2.9

Campaign Response:
  Avg Campaigns Accepted: 0.79
  Campaign Response Rate: 45.5%

CLUSTER 3
Size: 407 customers (18.5%)

Demographics:
  Average Income: $43,154
  Average Age:

In [77]:
# Assign descriptive names based on profiles
# Cluster 0: Age 56.9, Income $56k, 1 teen at home, moderate spending
# Cluster 1: Age 42.0, Income $30k, young kids, LOW spending  
# Cluster 2: Age 51.5, Income $76k, NO kids, HIGHEST spending
# Cluster 3: Age 54.7, Income $43k, 2+ children, low spending

cluster_names = {
    0: 'Mature Families w/ Teens',
    1: 'Young Families on Budget',
    2: 'Empty Nesters',
    3: 'Stretched Parents'
}

# Add cluster names to dataframe
df['Cluster_Name'] = df['Cluster'].map(cluster_names)

print("Cluster Names Assigned:")
print("="*80)
for cluster, name in cluster_names.items():
    count = (df['Cluster'] == cluster).sum()
    pct = count / len(df) * 100
    print(f"Cluster {cluster}: {name} - {count} customers ({pct:.1f}%)")

print("\n✓ Clusters profiled and named successfully!")

Cluster Names Assigned:
Cluster 0: Mature Families w/ Teens - 671 customers (30.4%)
Cluster 1: Young Families on Budget - 608 customers (27.6%)
Cluster 2: Empty Nesters - 519 customers (23.5%)
Cluster 3: Stretched Parents - 407 customers (18.5%)

✓ Clusters profiled and named successfully!


## 6. Visualizations

Now let's create compelling visualizations to showcase the differences between clusters.

In [78]:
# Cluster size distribution
cluster_counts = df['Cluster_Name'].value_counts()
total_count = len(df)
percentages = [(count / total_count * 100) for count in cluster_counts.values]

fig = px.bar(x=cluster_counts.index, y=cluster_counts.values,
             title='Customer Distribution Across Segments',
             labels={'x': 'Customer Segment', 'y': 'Number of Customers'},
             color=cluster_counts.values,
             color_continuous_scale='viridis',
             text=[f'{count}<br>({pct:.1f}%)' for count, pct in zip(cluster_counts.values, percentages)])
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=500, xaxis_tickangle=-45)
fig.show()


In [79]:
# Radar chart comparing cluster characteristics
from sklearn.preprocessing import MinMaxScaler

# Normalize features for better visualization
features_for_radar = ['Income', 'Age', 'MntTotal', 'NumStorePurchases', 'NumWebPurchases']
cluster_means = df.groupby('Cluster_Name')[features_for_radar].mean()

# Normalize to 0-1 scale
scaler_viz = MinMaxScaler()
cluster_means_normalized = pd.DataFrame(
    scaler_viz.fit_transform(cluster_means),
    columns=cluster_means.columns,
    index=cluster_means.index
)

fig = go.Figure()

for cluster_name in cluster_means_normalized.index:
    fig.add_trace(go.Scatterpolar(
        r=cluster_means_normalized.loc[cluster_name].values,
        theta=features_for_radar,
        fill='toself',
        name=cluster_name
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1])
    ),
    showlegend=True,
    title='Cluster Characteristics Comparison (Normalized)',
    height=600
)
fig.show()

In [80]:
# Income vs Total Spending by Cluster
fig = px.scatter(df, x='Income', y='MntTotal', color='Cluster_Name',
                 title='Customer Segments: Income vs Total Spending',
                 labels={'Income': 'Annual Income ($)', 
                        'MntTotal': 'Total Spending ($)',
                        'Cluster_Name': 'Segment'},
                 color_discrete_sequence=px.colors.qualitative.Bold,
                 hover_data=['Age', 'Recency', 'Kidhome', 'Teenhome'],
                 opacity=0.7)
fig.update_layout(height=600, legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02))
fig.show()

In [81]:
# Age vs Recency by Cluster
fig = px.scatter(df, x='Age', y='Recency', color='Cluster_Name',
                 title='Customer Segments: Age vs Purchase Recency',
                 labels={'Age': 'Age (years)', 
                        'Recency': 'Days Since Last Purchase',
                        'Cluster_Name': 'Segment'},
                 color_discrete_sequence=px.colors.qualitative.Bold,
                 hover_data=['Income', 'MntTotal'],
                 opacity=0.7)
fig.update_layout(height=600, legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02))
fig.show()

In [82]:
# Box plots comparing key metrics across clusters
fig = make_subplots(rows=2, cols=3,
                    subplot_titles=('Income', 'Total Spending', 'Age',
                                  'Recency', 'Store Purchases', 'Web Visits'))

colors = px.colors.qualitative.Bold[:len(df['Cluster_Name'].unique())]
cluster_order = sorted(df['Cluster_Name'].unique())

# Income
for i, cluster in enumerate(cluster_order):
    cluster_data = df[df['Cluster_Name'] == cluster]
    fig.add_trace(go.Box(y=cluster_data['Income'], name=cluster, 
                         marker_color=colors[i], showlegend=False), row=1, col=1)

# Total Spending
for i, cluster in enumerate(cluster_order):
    cluster_data = df[df['Cluster_Name'] == cluster]
    fig.add_trace(go.Box(y=cluster_data['MntTotal'], name=cluster, 
                         marker_color=colors[i], showlegend=False), row=1, col=2)

# Age
for i, cluster in enumerate(cluster_order):
    cluster_data = df[df['Cluster_Name'] == cluster]
    fig.add_trace(go.Box(y=cluster_data['Age'], name=cluster, 
                         marker_color=colors[i], showlegend=False), row=1, col=3)

# Recency
for i, cluster in enumerate(cluster_order):
    cluster_data = df[df['Cluster_Name'] == cluster]
    fig.add_trace(go.Box(y=cluster_data['Recency'], name=cluster, 
                         marker_color=colors[i], showlegend=False), row=2, col=1)

# Store Purchases
for i, cluster in enumerate(cluster_order):
    cluster_data = df[df['Cluster_Name'] == cluster]
    fig.add_trace(go.Box(y=cluster_data['NumStorePurchases'], name=cluster, 
                         marker_color=colors[i], showlegend=False), row=2, col=2)

# Web Visits
for i, cluster in enumerate(cluster_order):
    cluster_data = df[df['Cluster_Name'] == cluster]
    fig.add_trace(go.Box(y=cluster_data['NumWebVisitsMonth'], name=cluster, 
                         marker_color=colors[i], showlegend=False), row=2, col=3)

fig.update_layout(height=800, title_text="Key Metrics Comparison Across Segments", showlegend=False)
fig.show()

In [83]:
# Average spending by category for each cluster
product_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
product_labels = ['Wines', 'Fruits', 'Meat', 'Fish', 'Sweets', 'Gold']

spending_by_cluster = df.groupby('Cluster_Name')[product_cols].mean()
spending_by_cluster.columns = product_labels

# Create horizontal bar chart with food groups as traces
fig = go.Figure()

# Define colors for each food group
food_colors = {
    'Wines': '#8B0000',      # Dark red
    'Meat': '#CD5C5C',       # Indian red
    'Fish': '#4682B4',       # Steel blue
    'Fruits': '#FFD700',     # Gold
    'Sweets': '#FF69B4',     # Hot pink
    'Gold': '#DAA520'        # Goldenrod
}

for product in product_labels:
    fig.add_trace(go.Bar(
        name=product,
        y=spending_by_cluster.index,
        x=spending_by_cluster[product],
        orientation='h',
        marker_color=food_colors[product],
        text=spending_by_cluster[product].round(0),
        texttemplate='$%{text}',
        textposition='inside'
    ))

fig.update_layout(
    title='Average Spending by Product Category Across Segments',
    xaxis_title='Average Spending ($)',
    yaxis_title='Customer Segment',
    barmode='group',
    height=600,
    legend=dict(
        title='Product Category',
        orientation="v", 
        yanchor="top", 
        y=1, 
        xanchor="left", 
        x=1.02
    )
)
fig.show()

In [91]:
# Pie charts showing spending breakdown by food category for each segment
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{'type':'domain'}, {'type':'domain'}],
           [{'type':'domain'}, {'type':'domain'}]],
    subplot_titles=list(spending_by_cluster.index)
)

# Define consistent colors for food groups across all pie charts
food_colors_list = ['#8B0000', '#FFD700', '#CD5C5C', '#4682B4', '#FF69B4', '#DAA520']

row_col_positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

for idx, (segment_name, position) in enumerate(zip(spending_by_cluster.index, row_col_positions)):
    values = spending_by_cluster.loc[segment_name]
    
    fig.add_trace(go.Pie(
        labels=product_labels,
        values=values,
        name=segment_name,
        marker=dict(colors=food_colors_list),
        textinfo='label+percent',
        textposition='inside',
        hovertemplate='<b>%{label}</b><br>$%{value:.0f}<br>%{percent}<extra></extra>'
    ), row=position[0], col=position[1])

fig.update_layout(
    title_text='Food Category Spending Breakdown by Customer Segment',
    height=800,
    showlegend=True,
    legend=dict(
        title='Product Category',
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.05
    )
)

fig.show()

In [84]:
# Purchase channel preferences by cluster
channel_cols = ['NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
channel_labels = ['Deals', 'Web', 'Catalog', 'Store']

channels_by_cluster = df.groupby('Cluster_Name')[channel_cols].mean()
channels_by_cluster.columns = channel_labels

fig = go.Figure()

for segment in channels_by_cluster.index:
    fig.add_trace(go.Bar(
        name=segment,
        x=channel_labels,
        y=channels_by_cluster.loc[segment],
        text=channels_by_cluster.loc[segment].round(1),
        texttemplate='%{text}',
        textposition='outside'
    ))

fig.update_layout(
    title='Average Purchases by Channel Across Segments',
    xaxis_title='Purchase Channel',
    yaxis_title='Average Number of Purchases',
    barmode='group',
    height=600,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.show()

In [85]:
# Campaign acceptance by cluster
campaign_by_cluster = df.groupby('Cluster_Name')['AcceptedCmpOverall'].agg(['mean', 'sum'])
campaign_by_cluster['response_rate'] = (df.groupby('Cluster_Name')['AcceptedCmpOverall'].apply(lambda x: (x > 0).sum() / len(x) * 100))

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Avg Campaigns Accepted', 'Campaign Response Rate (%)'),
                    specs=[[{'type': 'bar'}, {'type': 'bar'}]])

fig.add_trace(go.Bar(
    x=campaign_by_cluster.index,
    y=campaign_by_cluster['mean'],
    marker_color='steelblue',
    text=campaign_by_cluster['mean'].round(2),
    textposition='outside'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=campaign_by_cluster.index,
    y=campaign_by_cluster['response_rate'],
    marker_color='coral',
    text=campaign_by_cluster['response_rate'].round(1),
    texttemplate='%{text}%',
    textposition='outside'
), row=1, col=2)

fig.update_xaxes(tickangle=-45, row=1, col=1)
fig.update_xaxes(tickangle=-45, row=1, col=2)
fig.update_layout(height=500, showlegend=False, title_text="Campaign Performance by Segment")
fig.show()

In [86]:
# Family composition by cluster
family_by_cluster = df.groupby('Cluster_Name')[['Kidhome', 'Teenhome']].mean()

fig = go.Figure()
fig.add_trace(go.Bar(name='Kids', x=family_by_cluster.index, y=family_by_cluster['Kidhome'],
                     marker_color='skyblue',
                     text=family_by_cluster['Kidhome'].round(2),
                     textposition='outside'))
fig.add_trace(go.Bar(name='Teens', x=family_by_cluster.index, y=family_by_cluster['Teenhome'],
                     marker_color='lightcoral',
                     text=family_by_cluster['Teenhome'].round(2),
                     textposition='outside'))

fig.update_layout(
    title='Average Household Composition by Segment',
    xaxis_title='Customer Segment',
    yaxis_title='Average Number of Children',
    barmode='group',
    height=500,
    xaxis_tickangle=-45,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.show()

In [87]:
# PCA visualization of clusters
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Cluster'] = df['Cluster_Name'].values
pca_df['Income'] = df['Income'].values
pca_df['MntTotal'] = df['MntTotal'].values

fig = px.scatter(pca_df, x='PC1', y='PC2', color='Cluster',
                 title=f'Customer Segments in 2D Space (PCA)<br>Explained Variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}',
                 labels={'PC1': f'First Principal Component ({pca.explained_variance_ratio_[0]:.1%})',
                        'PC2': f'Second Principal Component ({pca.explained_variance_ratio_[1]:.1%})',
                        'Cluster': 'Segment'},
                 color_discrete_sequence=px.colors.qualitative.Bold,
                 hover_data=['Income', 'MntTotal'],
                 opacity=0.7)
fig.update_layout(height=600, legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02))
fig.show()

In [88]:
print(f"\nTotal variance explained by 2 components: {pca.explained_variance_ratio_.sum():.1%}")


Total variance explained by 2 components: 61.9%


In [89]:
# Summary table with key metrics
summary_metrics = {
    'Segment': [],
    'Size': [],
    'Avg Income': [],
    'Avg Age': [],
    'Avg Spending': [],
    'Avg Children': [],
    'Campaign Response %': [],
    'Top Channel': []
}

for cluster_name in sorted(df['Cluster_Name'].unique()):
    cluster_data = df[df['Cluster_Name'] == cluster_name]
    
    summary_metrics['Segment'].append(cluster_name)
    summary_metrics['Size'].append(f"{len(cluster_data)} ({len(cluster_data)/len(df)*100:.1f}%)")
    summary_metrics['Avg Income'].append(f"${cluster_data['Income'].mean():,.0f}")
    summary_metrics['Avg Age'].append(f"{cluster_data['Age'].mean():.1f}")
    summary_metrics['Avg Spending'].append(f"${cluster_data['MntTotal'].mean():,.0f}")
    summary_metrics['Avg Children'].append(f"{(cluster_data['Kidhome'] + cluster_data['Teenhome']).mean():.2f}")
    summary_metrics['Campaign Response %'].append(f"{(cluster_data['AcceptedCmpOverall'] > 0).sum()/len(cluster_data)*100:.1f}%")
    
    # Find top channel
    channels = {
        'Store': cluster_data['NumStorePurchases'].mean(),
        'Web': cluster_data['NumWebPurchases'].mean(),
        'Catalog': cluster_data['NumCatalogPurchases'].mean()
    }
    top_channel = max(channels, key=channels.get)
    summary_metrics['Top Channel'].append(top_channel)

summary_df = pd.DataFrame(summary_metrics)

print("\n" + "="*120)
print("CUSTOMER SEGMENT SUMMARY")
print("="*120)
print(summary_df.to_string(index=False))
print("="*120)


CUSTOMER SEGMENT SUMMARY
                 Segment        Size Avg Income Avg Age Avg Spending Avg Children Campaign Response % Top Channel
           Empty Nesters 519 (23.5%)    $76,578    51.5       $1,343         0.10               45.5%       Store
Mature Families w/ Teens 671 (30.4%)    $56,608    56.9         $602         1.00               18.6%       Store
       Stretched Parents 407 (18.5%)    $43,154    54.7         $168         2.12                9.8%       Store
Young Families on Budget 608 (27.6%)    $30,485    42.0         $117         0.83                9.4%       Store


## 7. Key Insights and Recommendations

### Segment Profiles:

#### 1. Affluent Empty Nesters 💎
- **Who they are**: High-income, middle-aged customers (51.5 yrs) with virtually no children at home
- **Characteristics**: Highest spending ($1,343), especially on wines ($658) and meat ($462)
- **Key metrics**: 45.5% campaign response rate, low web visits (2.9/month), high store purchases (8.4)
- **Marketing strategy**: 
  - Premium product lines and exclusive offerings
  - Catalog and in-store VIP experiences
  - Wine clubs and gourmet food subscriptions
  - Loyalty programs with high-value rewards
  - Focus marketing budget here - highest ROI segment

#### 2. Mature Families with Teens 👨‍👩‍👦
- **Who they are**: Older customers (56.9 yrs) with teenagers at home, middle income
- **Characteristics**: Moderate spending ($602), wine-focused ($388), decent campaign response (18.6%)
- **Key metrics**: Balanced channel usage, 1 teen at home on average
- **Marketing strategy**: 
  - Family-sized premium products
  - Teen-friendly items alongside adult preferences
  - Back-to-school and holiday campaigns
  - Subscription services with family appeal
  - Store-focused promotions

#### 3. Stretched Parents 👨‍👩‍👧‍👦
- **Who they are**: Middle-aged (54.7 yrs) families with multiple children (2.12 avg), modest income
- **Characteristics**: Low spending ($168), most children per household, price-sensitive
- **Key metrics**: Low campaign response (9.8%), high web visits but low conversion
- **Marketing strategy**: 
  - Value bundles and family packs
  - Deal-focused campaigns
  - Budget-friendly product lines
  - Kid-focused marketing
  - Digital coupons and loyalty rewards

#### 4. Young Budget Families 💰
- **Who they are**: Youngest segment (42.0 yrs) with young kids, lowest income ($30,485)
- **Characteristics**: Lowest spending ($117), most price-conscious, limited engagement
- **Key metrics**: High web visits (6.8/month) but minimal purchases
- **Marketing strategy**: 
  - Entry-level pricing and promotions
  - Starter bundles for young families
  - Digital-first engagement (they're browsing!)
  - Email campaigns with deep discounts
  - Conversion optimization - they visit but don't buy

### Overall Recommendations:

1. **Differentiated Marketing**: Create segment-specific campaigns rather than one-size-fits-all
2. **Resource Allocation**: Prioritize **Affluent Empty Nesters** - 45.5% campaign response with highest spending
3. **Family Targeting**: Develop separate strategies for families with teens vs. young children
4. **Conversion Focus**: **Young Budget Families** visit the site frequently but don't buy - optimize pricing and promotions
5. **Premium Strategy**: Double down on **Affluent Empty Nesters** with wine, meat, and premium offerings
6. **Value Strategy**: Create compelling value propositions for the two stretched family segments
7. **Channel Strategy**: In-store experiences for affluent segments, digital deals for budget families
8. **Life Stage Marketing**: Align messaging with family life stage (young kids vs. teens vs. empty nest)

## 8. Export Clustered Data

In [90]:
# Export data with cluster assignments
output_path = "../data/processed/ifood_df_with_clusters.csv"
df.to_csv(output_path, index=False)

print(f"✓ Clustered data exported to: {output_path}")
print(f"  Shape: {df.shape}")
print(f"  New columns added: 'Cluster', 'Cluster_Name'")
print(f"\nCluster distribution:")
print(df['Cluster_Name'].value_counts())

✓ Clustered data exported to: ../data/processed/ifood_df_with_clusters.csv
  Shape: (2205, 41)
  New columns added: 'Cluster', 'Cluster_Name'

Cluster distribution:
Cluster_Name
Mature Families w/ Teens    671
Young Families on Budget    608
Empty Nesters               519
Stretched Parents           407
Name: count, dtype: int64
